In [ ]:
# API key need to be replacced

In [ ]:
# full

In [1]:
import os
import sys
import json
import time
import re
import pandas as pd
import chardet
import openai

def winpath(p: str) -> str:
    """Return absolute extended-length path on Windows (\\?\\ prefix)."""
    p = os.path.abspath(p)
    if sys.platform.startswith("win") and not p.startswith("\\\\?\\"):
        return "\\\\?\\" + p
    return p

def normalize_prompt_filename(name: str) -> str:
    name = (name or "").strip().replace("\\", "/").split("/")[-1]
    base = re.sub(r'(\.txt)+$', '', name, flags=re.IGNORECASE)
    return base + ".txt"

def resolve_txt_path(txt_root: str, project: str, commit: str, desired_filename: str) -> str | None:
    """Find the TXT under <root>/<project>/<commit>/ using exact/case-insensitive/contains match."""
    commit_dir = os.path.join(txt_root, project, commit)
    commit_dir_w = winpath(commit_dir)
    if not os.path.isdir(commit_dir_w):
        return None

    desired = normalize_prompt_filename(desired_filename)

  
    candidate = os.path.join(commit_dir, desired)
    if os.path.isfile(winpath(candidate)):
        return candidate

    
    try:
        candidates = [e.name for e in os.scandir(commit_dir_w) if e.is_file() and e.name.lower().endswith(".txt")]
    except FileNotFoundError:
        return None

    
    dlc = desired.lower()
    for fn in candidates:
        if fn.lower() == dlc:
            return os.path.join(commit_dir, fn)

    
    base = re.sub(r'\.txt$', '', desired, flags=re.IGNORECASE).lower()
    for fn in candidates:
        if base in fn.lower():
            return os.path.join(commit_dir, fn)

    return None

def read_text_file(path: str) -> str:
    """Read file with encoding detection (long-path safe)."""
    p = winpath(path)
    with open(p, "rb") as f:
        raw = f.read()
    enc = chardet.detect(raw).get("encoding") or "utf-8"
    with open(p, "r", encoding=enc, errors="replace") as f:
        return f.read()


client = openai.Client(api_key="Your key")

DATASET_CSV = r"G:\FULL_DATA_COLLECTED\FINAL_DATASET\with_version_info_corrected\merged_full_dataset_with_versions_and_claude_corrected.csv"
TXT_ROOT    = r"G:\FULL_DATA_COLLECTED\FINAL_DATASET\with_version_info_corrected\PackagedRecordsCleaned_corrected"
OUTPUT_DIR  = r"G:\FULL_DATA_COLLECTED\FINAL_DATASET\with_version_info_corrected\llm_generated_next_vulnerabilities"
os.makedirs(winpath(OUTPUT_DIR), exist_ok=True)

LLM_NAME = "gpt-4o"
RETRIES = 3
SLEEP_BETWEEN_CALLS_SEC = 8


ONE_SHOT_TEMPLATE = """You are a security analyst. For ONE record, read a minimal CSV row and its paired TXT file. Infer the “next vulnerability” and its CWE, then output EXACTLY ONE JSON object (no prose). Match the demo JSON’s keys, ordering, and types exactly. If a demo key is optional for your case, you may omit it—otherwise keep it.
— Brief CWE intro —
CWE (Common Weakness Enumeration) is a standardized catalog of software weakness types (e.g., “CWE-787: Out-of-bounds Write”). Identify the most fitting CWE ID and name for the next vulnerability.
— Problem Clarification —
• Next vulnerability: the new flaw unintentionally introduced by the earlier fix in “Previous Fix Details” and later corrected by “Future Candidate Details.” It is NOT the original flaw that the earlier fix targeted.
• Next CWE: the CWE (ID + name) that best categorizes that next vulnerability.
— Rules —
1) Use ONLY the provided inputs; no external knowledge.
2) Read BOTH diffs/messages (Previous Fix + Future Candidate). Treat any “Reasoning” block as a hint: adopt if supported by code; otherwise include (optionally) as a lower-probability alternate.
3) Map the next CWE ONLY if clearly supported by code/messages (bounds checks, signedness, endianness, reachable assert, NULL handling, concurrency, lifetime/UAF, error handling, etc.). If unclear, set "mapping_status": "not_identified" and leave the main CWE fields empty strings.
4) Write ONE generic task→action one-liner (≤ 25 words) using the template: “Fixing X by Y can cause Z.” No project/library names.
5) Probabilities in [0,1] with two decimals. Main ≥ each alternate. Up to TWO alternates. They need not sum to 1.
6) OUTPUT: Return ONLY one JSON object, following the demo’s keys and order exactly.
==================== ONE-SHOT DEMO ====================
[INPUT A: Minimal CSV row]
project=bdwgc
commit=e10c1eb9908c2774c16b3148b30d2f3823d66a9a
initial_cwe=CWE-189
initial_cve=CVE-2012-2673
[INPUT B: Paired TXT content]
### Previous Fix Details
Commit Message: Fix calloc() overflow
Code Changes (Diff Format):
+ #ifndef SIZE_MAX
+ #define SIZE_MAX (~(size_t)0)
+ #endif
void * calloc(size_t n, size_t lb) {{
+  if (lb && n > SIZE_MAX / lb) return NULL;
  ...
}}
### Future Candidate Details
Commit Message: Ensure oom_fn callback executed on out-of-memory in calloc
Code Changes (Diff Format):
if ((lb | n) > GC_SQRT_SIZE_MAX && lb && n > GC_SIZE_MAX / lb)
-  return NULL;
+  return (*GC_get_oom_fn())(GC_SIZE_MAX); /* n*lb overflow */
===== Reasoning =====
Earlier fix returned NULL on overflow; later fix routes overflow to oom_fn.
[EXPECTED OUTPUT JSON]
{{
  "project": "bdwgc",
  "commit": "e10c1eb9908c2774c16b3148b30d2f3823d66a9a",
  "initial_cwe": "CWE-189",
  "initial_cve": "CVE-2012-2673",
  "mapping_status": "mapped",
  "next_cwe_id": "CWE-703",
  "next_cwe_name": "Improper Check or Handling of Exceptional Conditions",
  "one_line_causal": "Fixing size-multiplication overflow by early NULL returns can cause inconsistent failure handling across callers.",
  "probability": 0.78,
  "alternates": [
    {{
      "id": "CWE-388",
      "name": "Improper Error Handling",
      "one_line_causal": "Fixing overflow via early NULL returns can cause generic error paths to be bypassed.",
      "probability": 0.15
    }},
    {{
      "id": "CWE-476",
      "name": "NULL Pointer Dereference",
      "one_line_causal": "Fixing overflow by returning NULL can cause downstream NULL dereference when callers omit checks.",
      "probability": 0.07
    }}
  ]
}}
==================== YOUR TASK ====================
[INPUT A: Minimal CSV row]
project={project}
commit={commit}
initial_cwe={initial_cwe}
initial_cve={initial_cve}
[INPUT B: Paired TXT content]
{txt_content}
— Produce ONLY the JSON object, matching the demo’s format exactly. No prose.
"""

def build_prompt(project: str, commit: str, initial_cwe: str, initial_cve: str, txt_content: str) -> str:
    initial_cwe = initial_cwe if isinstance(initial_cwe, str) and initial_cwe.strip() else ""
    initial_cve = initial_cve if isinstance(initial_cve, str) and initial_cve.strip() else ""
    return ONE_SHOT_TEMPLATE.format(
        project=project,
        commit=commit,
        initial_cwe=initial_cwe,
        initial_cve=initial_cve,
        txt_content=txt_content
    )

def clean_and_parse_json(reply_text: str):
    txt = reply_text.strip()
    if txt.startswith("```"):
        txt = re.sub(r"^```(?:json)?\s*", "", txt)
        txt = re.sub(r"\s*```$", "", txt.strip())
    return json.loads(txt)


df = pd.read_csv(winpath(DATASET_CSV))
needed_cols = ["Project", "commit", "CWE ID", "CVE ID", "Prompt File"]
missing = [c for c in needed_cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")
df = df[needed_cols].copy()   # <-- no head(); process everything


results, raw_jsonl, errors = [], [], []

for idx, row in df.iterrows():
    project = str(row["Project"]).strip()
    commit = str(row["commit"]).strip()
    prompt_file_raw = str(row["Prompt File"]).strip()
    prompt_file = normalize_prompt_filename(prompt_file_raw)
    initial_cwe = str(row["CWE ID"]).strip() if pd.notna(row["CWE ID"]) else ""
    initial_cve = str(row["CVE ID"]).strip() if pd.notna(row["CVE ID"]) else ""

    txt_path = resolve_txt_path(TXT_ROOT, project, commit, prompt_file)
    if not txt_path:
        errors.append({
            "project": project, "commit": commit, "prompt_file_csv": prompt_file_raw,
            "prompt_file_norm": prompt_file, "tried_path": os.path.join(TXT_ROOT, project, commit, prompt_file),
            "error_type": "File Not Found", "details": "Could not resolve TXT path"
        })
        print(f"❌ SKIPPED (read error): {project}/{commit} :: {prompt_file_raw} → {prompt_file} — not found")
        continue

    try:
        txt_content = read_text_file(txt_path)  
    except Exception as e:
        errors.append({
            "project": project, "commit": commit, "prompt_file_csv": prompt_file_raw,
            "prompt_file_norm": prompt_file, "tried_path": txt_path,
            "error_type": "File Read Error", "details": str(e)
        })
        print(f"❌ SKIPPED (read error): {project}/{commit} :: {prompt_file_raw} → {prompt_file} — {e}")
        continue

    prompt = build_prompt(project, commit, initial_cwe, initial_cve, txt_content)

    response_json = None
    for attempt in range(RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_NAME,
                messages=[
                    {"role": "system", "content": "You are a cybersecurity expert."},
                    {"role": "user", "content": prompt}
                ]
            )
            reply = resp.choices[0].message.content.strip()
            response_json = clean_and_parse_json(reply)
            break
        except Exception as e:
            if attempt == RETRIES - 1:
                errors.append({
                    "project": project, "commit": commit, "prompt_file": prompt_file,
                    "error_type": "API Error", "details": str(e)
                })
            time.sleep(10)

    if response_json is not None:
        alts = response_json.get("alternates", []) or []
        raw_jsonl.append(response_json)
        results.append({
            "Project": project,
            "commit": commit,
            "Prompt File (CSV)": prompt_file_raw,
            "Prompt File (normalized)": prompt_file,
            "TXT Path Used": txt_path,
            "output_json": json.dumps(response_json, ensure_ascii=False),
            "mapping_status": response_json.get("mapping_status", ""),
            "next_cwe_id": response_json.get("next_cwe_id", ""),
            "next_cwe_name": response_json.get("next_cwe_name", ""),
            "one_line_causal": response_json.get("one_line_causal", ""),
            "probability": response_json.get("probability", ""),
            "alt1_id": alts[0].get("id", "") if len(alts) > 0 else "",
            "alt1_name": alts[0].get("name", "") if len(alts) > 0 else "",
            "alt1_probability": alts[0].get("probability", "") if len(alts) > 0 else "",
            "alt2_id": alts[1].get("id", "") if len(alts) > 1 else "",
            "alt2_name": alts[1].get("name", "") if len(alts) > 1 else "",
            "alt2_probability": alts[1].get("probability", "") if len(alts) > 1 else "",
        })

        print(f"✅ Success {len(results)}/{len(df)} — {project}/{commit}")

    time.sleep(SLEEP_BETWEEN_CALLS_SEC)


jsonl_path = os.path.join(OUTPUT_DIR, f"{LLM_NAME}_results.jsonl")
with open(winpath(jsonl_path), "w", encoding="utf-8") as f:
    for obj in raw_jsonl:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

summary_xlsx = os.path.join(OUTPUT_DIR, f"{LLM_NAME}_summary.xlsx")
pd.DataFrame(results).to_excel(winpath(summary_xlsx), index=False)

if errors:
    err_xlsx = os.path.join(OUTPUT_DIR, f"{LLM_NAME}_errors.xlsx")
    pd.DataFrame(errors).to_excel(winpath(err_xlsx), index=False)

print(f"\n✅ Completed.\n🧾 JSONL: {jsonl_path}\n📊 Summary: {summary_xlsx}")


✅ Success 1/469 — bento4/8922f0dcc30b56936e2b12e819e52b86e3300bfa
✅ Success 2/469 — bento4/ab4d641e395b09c7bb86acfe243a8b986609d72c
✅ Success 3/469 — bento4/e13b22fe41f87db766a061d5003d4019a3ed5c18
✅ Success 4/469 — chakracore/065b7978c40ded35c356ced6cd922a40156c9c46
✅ Success 5/469 — chakracore/402f3d967c0a905ec5b9ca9c240783d3f2c15724
✅ Success 6/469 — collabcal/b80f6d1893607c99e5113967592417d0fe310ce6
✅ Success 7/469 — espruino/34fd6cd10f6cf21bc79c5e099f38c3d2afbe5902
✅ Success 8/469 — ffmpeg/189ff4219644532bdfa7bab28dfedaee4d6d4021
✅ Success 9/469 — ffmpeg/547d690d676064069d44703a1917e0dab7e33445
✅ Success 10/469 — ffmpeg/821a5938d100458f4d09d634041b05c860554ce0
✅ Success 11/469 — ffmpeg/b97a4b658814b2de8b9f2a3bce491c002d34de31
✅ Success 12/469 — freerdp/d6f9d33a7db0b346195b6a15b5b99944ba41beee
✅ Success 13/469 — freertos-kernel/47338393f1f79558f6144213409f09f81d7c4837
✅ Success 14/469 — imagemagick/3b0fe05cddd8910f84e51b4d50099702ea45ba4a
✅ Success 15/469 — imagemagick/4b9eeffe5b15